In [14]:
import numpy as np
import matplotlib.pyplot as plt
import mne
import pandas as pd
from pathlib import Path

Fs = 500
DATA_FOLDER = Path("data/calculia")
RESULTS_FOLDER = Path("results")

# Channel names
CHANNELS = {
    "ANT_FRONTAL" : ["Fp1", "Fp2"],
    "FRONTAL" : ["F3", "F4", "Fz", "F7", "F8"],
    "CENTRAL" : ["C3", "C4", "Cz"],
    "PARIETAL" : ["P3", "P4", "Pz"],
    "OCCIPITAL" : ["O1", "O2"],
    "TEMPORAL" : ["T3", "T4", "T5", "T6"],
}

all_channels = []
for chs in CHANNELS.values():
    all_channels += chs
CHANNELS["ALL_CHANNELS"] = all_channels

In [2]:
BG_COLOR = "#f5f5f0"

In [3]:
from _functions import PCE_by_band

PCEs = []

for sbj_id in range(36):
    sbj = f"Subject{sbj_id:02d}"
    rest_raw = mne.io.read_raw_edf(DATA_FOLDER / f"{sbj}_1.edf", verbose=False)
    calc_raw = mne.io.read_raw_edf(DATA_FOLDER / f"Subject{sbj_id:02d}_2.edf", verbose=False)


    for name, channels in CHANNELS.items():           
        rest_np = np.array(rest_raw[["EEG " + ch for ch in channels]][0])
        rest = PCE_by_band(rest_np, Fs = Fs)
        for result in rest:
            PCEs.append({
                "Subject" : sbj,
                "Location" : name,
                "Band" : result["band"],
                "Rest" : 1,
                "PCE" : result["PCE"]
            }) 

        calc_np = np.array(calc_raw[["EEG " + ch for ch in channels]][0])
        calc = PCE_by_band(rest_np, Fs = Fs)
        for result in calc:
                PCEs.append({
                    "Subject" : sbj,
                    "Location" : name,
                    "Band" : result["band"],
                    "Rest" : 0,
                    "PCE" : result["PCE"]
                }) 


df_results = pd.DataFrame(PCEs)
df_results.head()

,Subject,Location,Band,Rest,PCE
0,Subject00,ANT_FRONTAL,delta,1,0.271118
1,Subject00,ANT_FRONTAL,theta,1,0.209871
2,Subject00,ANT_FRONTAL,alpha,1,0.104924
3,Subject00,ANT_FRONTAL,beta,1,0.207345
4,Subject00,ANT_FRONTAL,gamma,1,0.301815


In [15]:
df_results.to_csv(RESULTS_FOLDER / "arithmetic_pce.csv")

## Merge user Information

In [4]:
user_df = pd.read_csv(DATA_FOLDER / "subject-info.csv")
df = df_results.merge(user_df, on = "Subject")
df["Male"] = df.Gender.map({"M" : 1, "F" : 0})
df = df.drop(columns=["Recording year", "Number of subtractions", "Gender"])

In [22]:
good_counters_mask = df["Count quality"] == 1
rest_mask = df["Rest"] == 1

In [ ]:
# fig, axes = plt.subplots(2, 1, figsize = (15, 8), facecolor = BG_COLOR)
# barWidth = 0.25

# br1 = np.arange(df.Location.nunique()) 
# br2 = [x + barWidth for x in br1] 

# data = df[good_counters_mask].groupby("Location")["PCE"].mean()
# axes[0].barh(br1, data.values)#, width = barWidth)
# data = df[~good_counters_mask].groupby("Location")["PCE"].mean()
# axes[0].barh(br2, data.values)#, width = barWidth)
# axes[0].set_yticklabels(data.index)

# data = df[good_counters_mask].groupby("Band")["PCE"].mean()
# axes[1].barh(data.index, data.values)
# data = df[~good_counters_mask].groupby("Band")["PCE"].mean()
# axes[1].barh(data.index, data.values)

In [25]:
from scipy.stats import mannwhitneyu
from _functions import cohen_d

res = []

for rest in [0, 1]:
    rest_mask_cur = rest_mask
    if rest == 0:
        rest_mask_cur = ~ rest_mask_cur

    for ch in list(CHANNELS.keys()):
        channel_mask = df["Location"] == ch
        for band in df["Band"].unique():
            band_mask = df["Band"] == band
            good_counters = df.loc[good_counters_mask & channel_mask & band_mask,
                                   "PCE"].values
            bad_counters = df.loc[~good_counters_mask & channel_mask & band_mask,
                                   "PCE"].values
            _, p = mannwhitneyu(good_counters, bad_counters)
            effect_size = cohen_d(good_counters, bad_counters)

            res.append({
                "Location" : ch,
                "Band" : band,
                "Rest" : rest,
                "p" : p,
                "cohen_d" : effect_size
            })

df_comparisons = pd.DataFrame(res)
df_comparisons.to_csv(RESULTS_FOLDER / "good vs bad counters.csv")

ImportError: cannot import name 'cohen_d' from '_functions' (c:\Users\gabri\OneDrive\Desktop\EEG-Synchronization\_functions.py)

In [ ]:
fig, ax = plt.subplots(12, 5, face_color = BG_COLOR)

ax.scatter(df_comparisons["cohen_d"], -np.log10(df_comparisons["p"]))

plt.show()